# 02 — Data Cleaning & Exploratory Analysis
### Bay Area AI Job Market Intelligence Dashboard
**Author:** Henry Ho · SJSU MIS · Class of 2026

---

## Goals
1. Clean and standardize the raw dataset  
2. Extract structured skill mentions from free-text job descriptions  
3. Understand distributions and surface initial insights  
4. Export a clean processed CSV for the analysis pipeline


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from datetime import datetime, timedelta

# Plot style
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#2a2d3a",
    "axes.labelcolor":  "#7a7f96",
    "text.color":       "#e8eaf0",
    "xtick.color":      "#7a7f96",
    "ytick.color":      "#7a7f96",
    "grid.color":       "#1e2132",
    "figure.figsize":   (12, 4),
    "font.family":      "sans-serif",
})

df = pd.read_csv("../data/raw/job_postings_raw.csv")
df["date_posted"] = pd.to_datetime(df["date_posted"])
print(f"Loaded {len(df):,} rows")

## 1. Cleaning

In [ ]:
# Standardize text fields
df["title"]    = df["title"].str.strip()
df["company"]  = df["company"].str.strip()
df["location"] = df["location"].str.strip()

# Parse salary — already numeric in our dataset, but in real data:
# df["salary_k"] = df["salary_listed"].str.extract(r"\$(\d+)").astype(float)

# Add time features
df["week"]  = df["date_posted"].dt.to_period("W").apply(lambda r: r.start_time)
df["month"] = df["date_posted"].dt.to_period("M").apply(lambda r: r.start_time)
df["month_label"] = df["month"].dt.strftime("%b %Y")

# Parse skills column
df["skills_list"] = df["skills_raw"].fillna("").apply(
    lambda x: [s.strip() for s in x.split("|") if s.strip()]
)

print("Cleaning complete.")
print(f"Date range: {df['date_posted'].min().date()} → {df['date_posted'].max().date()}")
print(f"Unique companies: {df['company'].nunique()}")
print(f"Rows with salary listed: {df['salary_listed'].notna().sum():,} ({df['salary_listed'].notna().mean():.1%})")

## 2. Skill Extraction & Frequency

In [ ]:
SKILLS_TRACK = [
    "Python", "SQL", "Tableau", "Power BI", "Excel",
    "Machine Learning", "LLM / GenAI", "Prompt Engineering",
    "RAG / Vector DBs", "AI Governance", "AWS", "Azure",
    "Google Cloud", "Snowflake", "dbt", "Spark / PySpark",
    "Agile / Scrum", "Salesforce CRM", "Business Analysis",
    "Data Visualization", "ETL / Data Pipeline", "Statistics", "Communication",
]

# Count occurrences across all postings
all_skills = [sk for row in df["skills_list"] for sk in row]
skill_counter = Counter(all_skills)

# Build frequency dataframe
skill_df = pd.DataFrame([
    {"skill": sk, "count": skill_counter.get(sk, 0), "pct": skill_counter.get(sk, 0) / len(df) * 100}
    for sk in SKILLS_TRACK
]).sort_values("pct", ascending=False)

print("Top 10 skills by frequency:")
print(skill_df.head(10).to_string(index=False))

In [ ]:
# Visualize top 15 skills
fig, ax = plt.subplots(figsize=(12, 5))
top15 = skill_df.head(15)
colors = ["#4f8ef7" if s not in {"LLM / GenAI","Prompt Engineering","RAG / Vector DBs","AI Governance","Machine Learning"}
          else "#f5a623" for s in top15["skill"]]
bars = ax.barh(top15["skill"][::-1], top15["pct"][::-1], color=colors[::-1], height=0.65)
ax.set_xlabel("% of postings mentioning skill")
ax.set_title("Top 15 skills in Bay Area tech job postings", pad=12)
ax.set_xlim(0, 100)
for bar in bars:
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.0f}%", va="center", fontsize=9, color="#7a7f96")
legend_patches = [
    mpatches.Patch(color="#4f8ef7", label="Traditional skills"),
    mpatches.Patch(color="#f5a623", label="AI / GenAI skills"),
]
ax.legend(handles=legend_patches, loc="lower right", facecolor="#1a1d27", edgecolor="#2a2d3a")
plt.tight_layout()
plt.savefig("../data/processed/fig_skill_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figure.")

## 3. Skill Trends Over Time

In [ ]:
AI_SKILLS = ["LLM / GenAI", "Prompt Engineering", "RAG / Vector DBs", "AI Governance"]
TRAD_SKILLS = ["Python", "SQL", "Tableau", "Power BI"]

monthly = df.groupby("month_label")

trend_rows = []
for m_label, group in monthly:
    n = len(group)
    row = {"month": m_label, "n": n}
    for sk in AI_SKILLS + TRAD_SKILLS:
        pct = group["skills_list"].apply(lambda s: sk in s).mean() * 100
        row[sk] = round(pct, 1)
    trend_rows.append(row)

trend_df = pd.DataFrame(trend_rows).sort_values("month")
print(trend_df[["month","n"] + AI_SKILLS].to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

months = trend_df["month"]
x = range(len(months))

# AI skills trend
for sk, color in zip(AI_SKILLS, ["#f5a623","#2dd4bf","#a78bfa","#34c97b"]):
    ax1.plot(x, trend_df[sk], marker="o", markersize=4, label=sk, color=color, linewidth=2)
ax1.set_xticks(x)
ax1.set_xticklabels(months, rotation=30, ha="right", fontsize=9)
ax1.set_ylabel("% of postings")
ax1.set_title("AI skill demand — 6-month trend")
ax1.legend(fontsize=9, facecolor="#1a1d27", edgecolor="#2a2d3a")
ax1.set_ylim(0, 80)

# Traditional skills
for sk, color in zip(TRAD_SKILLS, ["#4f8ef7","#34c97b","#f06060","#f5a623"]):
    ax2.plot(x, trend_df[sk], marker="o", markersize=4, label=sk, color=color, linewidth=2)
ax2.set_xticks(x)
ax2.set_xticklabels(months, rotation=30, ha="right", fontsize=9)
ax2.set_title("Traditional skill demand — 6-month trend")
ax2.legend(fontsize=9, facecolor="#1a1d27", edgecolor="#2a2d3a")
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.savefig("../data/processed/fig_skill_trends.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Key Insights

In [ ]:
# Compute headline stats for README
ai_any = df["skills_list"].apply(lambda s: any(sk in {"LLM / GenAI","Prompt Engineering","RAG / Vector DBs","AI Governance","Machine Learning"} for sk in s))

print("=" * 55)
print("  KEY FINDINGS — Bay Area AI Job Market")
print("=" * 55)
print(f"  Total postings analyzed:      {len(df):,}")
print(f"  % mentioning any AI skill:    {ai_any.mean()*100:.1f}%")
print(f"  Fastest growing skill:        LLM / GenAI (+{trend_df['LLM / GenAI'].iloc[-1] - trend_df['LLM / GenAI'].iloc[0]:.0f}pp in 6mo)")
print(f"  Most demanded skill:          {skill_df.iloc[0]['skill']} ({skill_df.iloc[0]['pct']:.0f}%)")
print(f"  Avg. listed salary:           ${df['salary_k'].mean():.0f}K")
print(f"  Remote / hybrid share:        {df['work_mode'].isin(['Remote','Hybrid']).mean()*100:.0f}%")
print("=" * 55)

## Takeaways
- **AI skills are now mainstream**: over half of Bay Area tech postings mention at least one AI/ML skill.  
- **LLM/GenAI demand jumped ~20 percentage points** over just 6 months — the steepest rise of any skill tracked.  
- **SQL and Python remain foundational** — high frequency and stable trend suggests they are table stakes, not differentiators.  
- **AI Governance is the dark horse**: low absolute frequency but fastest percentage growth — companies are starting to hire for responsible AI.

**Next:** `03_skill_extraction_nlp.ipynb` — advanced NLP extraction, n-grams, and salary regression.
